# Private ES Index Only

**Routing pattern:**
- WRITE: `[elasticsearch_private]` — only geoid tokens stored; no attributes persisted anywhere
- READ: `[elasticsearch_private]`
- SEARCH: `[elasticsearch_private]`
- METADATA: `[postgresql]` — collection metadata stays in PG registry

Use this for maximum data privacy — only geoid tokens and spatial coordinates are stored; no item attributes are persisted.

In [ ]:
import httpx

import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=False)
BASE = os.environ.get("DYNASTORE_BASE_URL") or "http://localhost:8080"

# Auto-provision DYNASTORE_TOKEN via IDP client_credentials if not already set
if not os.environ.get("DYNASTORE_TOKEN"):
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                os.environ["DYNASTORE_TOKEN"] = _r.json().get("access_token", "")
        except Exception:
            pass
CATALOG_ID = "demo-private-only"
COLLECTION_ID = "anon-locations"

client = httpx.AsyncClient(base_url=BASE, timeout=30)

def _check(r, label=""):
    status = r.status_code
    ok = status < 400
    suffix = "" if ok else f" — {r.text[:300]}"
    print(f"{label + ': ' if label else ''}{status}{suffix}")
    return r

print("Client ready")

In [ ]:
import asyncio

r = await client.post("/stac/catalogs", json={"id": CATALOG_ID, "title": "Private ES Only Demo", "description": "Private ES Only Demo catalog."})
_check(r)
# Wait for provisioning to complete before creating collections
for _ in range(30):
    r = await client.get(f"/stac/catalogs/{CATALOG_ID}")
    if r.status_code == 200 and r.json().get("provisioning_status") == "ready":
        print("Catalog ready")
        break
    await asyncio.sleep(1)
else:
    print("Warning: catalog still provisioning after 30s")

In [ ]:
r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLLECTION_ID, "title": "Anonymous Locations",
    "description": "Anonymous Locations collection.",
    "license": "proprietary",
    "extent": {"spatial": {"bbox": [[-180, -90, 180, 90]]}, "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]}},
})
_check(r)

In [ ]:
routing = {
    "operations": {
        "WRITE": [{"driver_ref": "ItemsElasticsearchPrivateDriver", "on_failure": "fatal"}],
        "READ": [{"driver_ref": "ItemsElasticsearchPrivateDriver"}],
        "SEARCH": [{"driver_ref": "ItemsElasticsearchPrivateDriver"}],
        "METADATA": [{"driver_ref": "items_postgresql_driver"}]
    }
}
r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/plugins/collection_routing_config",
    json=routing,
)
_check(r)

In [ ]:
features = [
    {
        "type": "Feature",
        "id": f"loc-{i:03d}",
        "geometry": {"type": "Point", "coordinates": [12.5 + i * 0.01, 41.9 + i * 0.01]},
        "bbox": [12.5 + i * 0.01, 41.9 + i * 0.01, 12.5 + i * 0.01, 41.9 + i * 0.01],
        "properties": {"pii_field": f"secret-{i}"}  # will NOT be stored
    }
    for i in range(5)
]
r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    json={"type": "FeatureCollection", "features": features},
)
_check(r, "5 items inserted (only geoids stored in ES)")

In [ ]:
r = await client.get(f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items?limit=10")
data = r.json()
print(f"Items: {len(data.get('features', []))}")
for f in data.get("features", []):
    # pii_field is absent — only geoid token returned
    assert "pii_field" not in f.get("properties", {}), "PII leaked!"
    print(" ", f.get("id"), "(no pii_field — correct)")

In [ ]:
r = await client.delete(f"/stac/catalogs/{CATALOG_ID}?force=true")
_check(r, "Cleanup")
await client.aclose()